# Human time-travel matrix (release×release) — analysis + manuscript figures

This notebook consumes the cached grid produced by:

- `idtrack/reproducibility/experiments/experiment_time_travel_matrix/00_build_time_travel_matrix_cache.ipynb`

and generates **manuscript-ready** multi-panel figures + small summary tables.

## Core idea to market

Most mapping tools implicitly choose “current” database state. IDTrack instead makes `from_release` and `to_release` explicit, enabling a reviewer-proof claim:

> identifier mapping is a reproducibility contract across **namespace × release × assembly**.

This notebook makes the **time axis** visually undeniable via square heatmaps.

## Outputs

- `_outputs/_publication/figures/fig_time_travel_matrix_human.pdf` (multi-panel heatmaps)
- `_outputs/_publication/figures/fig_time_travel_delta_curves.pdf` (distance-to-drift curves)
- `_outputs/_publication/figures/fig_time_travel_asymmetry_human.pdf` (directionality / irreversibility)
- `_outputs/_publication/figures/fig_time_travel_runtime_human.pdf` (runtime heatmaps)
- `_outputs/_publication/figures/fig_time_travel_release_difficulty_human.pdf` (source/target difficulty)
- `_outputs/_publication/tables/time_travel_matrix_delta_summary.csv`
- `_outputs/_publication/figures/fig_time_travel_directionality_stability_human.pdf` (past vs future + stability indices)
- `_outputs/_publication/tables/time_travel_release_stability_index.csv`
- `_outputs/_publication/tables/time_travel_pair_extremes.csv`

All figures/tables are also mirrored under:

- `idtrack/reproducibility/experiments/_outputs/time_travel_matrix/`


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import os
import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    atomic_write_dataframe_csv,
    notebook_context,
    read_json,
    read_pickle,
    save_figure,
)

ctx = notebook_context("time_travel_matrix", start=REPO_ROOT)
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures
MANUSCRIPT_TABLES = ctx.manuscript_tables

EXPERIMENT_OUTPUTS = ctx.experiment_outputs
EXPERIMENT_FIGURES = EXPERIMENT_OUTPUTS / "figures"
EXPERIMENT_TABLES = EXPERIMENT_OUTPUTS / "tables"
EXPERIMENT_FIGURES.mkdir(parents=True, exist_ok=True)
EXPERIMENT_TABLES.mkdir(parents=True, exist_ok=True)

print("CACHE_DIR:", CACHE_DIR)
print("EXPERIMENT_OUTPUTS:", EXPERIMENT_OUTPUTS)
print("MANUSCRIPT_FIGURES:", MANUSCRIPT_FIGURES)
print("MANUSCRIPT_TABLES:", MANUSCRIPT_TABLES)


In [ ]:
# -------------------- Load latest cached grid --------------------

candidates = list(CACHE_DIR.glob("time_travel_matrix_grid_*.pickle"))
if not candidates:
    raise FileNotFoundError(
        f"No grid cache found under {CACHE_DIR}. Run: 00_build_time_travel_matrix_cache.ipynb"
    )

RESULTS_PKL = max(candidates, key=lambda p: p.stat().st_mtime)
fp = RESULTS_PKL.stem.split("_")[-1]

PARAMS_JSON = CACHE_DIR / f"time_travel_matrix_params_{fp}.json"
POOLS_JSON = CACHE_DIR / f"time_travel_matrix_pools_{fp}.json"

grid = read_pickle(RESULTS_PKL)
params = read_json(PARAMS_JSON) if PARAMS_JSON.exists() else {}

print("Using:", RESULTS_PKL)
print("Params:", PARAMS_JSON if PARAMS_JSON.exists() else "(missing)")
print("Pools:", POOLS_JSON if POOLS_JSON.exists() else "(missing)")
display(pd.DataFrame([params]) if params else pd.DataFrame())
display(grid.head())


In [ ]:
# -------------------- Derive fractions + aggregate over bootstraps --------------------

df = grid.copy()
den = df["total"].replace(0, np.nan)

# Core outcome fractions
df["frac_1_to_0"] = df["1_to_0"] / den
df["frac_1_to_1_tdm"] = df["1_to_1_tdm"] / den
df["frac_1_to_1_atm"] = df["1_to_1_atm"] / den
df["frac_1_to_n_tdm"] = df["1_to_n_tdm"] / den
df["frac_1_to_n_atm"] = df["1_to_n_atm"] / den
df["frac_1_to_1_total"] = df["1_to_1_total"] / den
df["frac_1_to_n_total"] = df["1_to_n_total"] / den

# Diagnostic aggregates
df["frac_atm_total"] = (df["1_to_1_atm"] + df["1_to_n_atm"]) / den
df["frac_tdm_total"] = (df["1_to_1_tdm"] + df["1_to_n_tdm"]) / den
df["frac_changed_any"] = (df["changed_1_to_1"] + df["changed_1_to_n"]) / den

group_cols = ["from_release", "to_release", "final_database"]
mean = df.groupby(group_cols, as_index=False).mean(numeric_only=True)
std = df.groupby(group_cols, as_index=False).std(numeric_only=True).rename(columns=lambda c: f"std_{c}" if c not in group_cols else c)

agg = mean.merge(std, on=group_cols, how="left")
display(agg.head())


In [ ]:
# -------------------- Figure: release×release heatmaps (multi-panel) --------------------

def _heatmap(ax, mat: pd.DataFrame, *, title: str, cmap: str = "viridis"):
    if sns is not None:
        sns.heatmap(
            mat,
            ax=ax,
            vmin=0,
            vmax=1,
            cmap=cmap,
            cbar=False,
            square=True,
            linewidths=0.25,
            linecolor=MANUSCRIPT_COLORS["grid"],
        )
    else:
        ax.imshow(mat.values, vmin=0, vmax=1, cmap=cmap)
        ax.set_xticks(np.arange(len(mat.columns)))
        ax.set_yticks(np.arange(len(mat.index)))
        ax.set_xticklabels([str(c) for c in mat.columns], rotation=45, ha="right")
        ax.set_yticklabels([str(i) for i in mat.index], rotation=0)

    # seaborn heatmaps already set tick labels; just rotate for readability
    if sns is not None:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    ax.set_title(title)
    ax.set_xlabel("to_release")
    ax.set_ylabel("from_release")


def _pivot(metric: str, *, final_db: str) -> pd.DataFrame:
    sub = agg[agg["final_database"] == final_db]
    if sub.empty:
        raise ValueError(f"No rows for final_database={final_db!r}")
    mat = sub.pivot(index="from_release", columns="to_release", values=metric)
    mat = mat.sort_index().sort_index(axis=1)
    return mat


# Focus the primary manuscript figure on backbone + HGNC.
rows = [
    ("Ensembl gene", "Ensembl backbone"),
    ("HGNC Symbol", "External match (HGNC)"),
]

metrics = [
    ("frac_1_to_1_tdm", "1→1 (TDM)", "viridis"),
    ("frac_1_to_n_tdm", "1→n (TDM)", "viridis"),
    ("frac_1_to_0", "1→0", "Reds"),
    ("frac_atm_total", "ATM fallback", "Oranges"),
]

fig, axes = plt.subplots(len(rows), len(metrics), figsize=(4.4 * len(metrics), 4.0 * len(rows)), constrained_layout=True)
if len(rows) == 1:
    axes = np.array([axes])

for r_idx, (final_db, row_title) in enumerate(rows):
    for c_idx, (metric, col_title, cmap) in enumerate(metrics):
        ax = axes[r_idx, c_idx]
        mat = _pivot(metric, final_db=final_db)
        _heatmap(ax, mat, title=f"{row_title} — {col_title}", cmap=cmap)

written = save_figure(fig, "fig_time_travel_matrix_human.pdf", ctx, formats=("pdf",))
print("Saved:", written["pdf"])


In [ ]:
# -------------------- Figure: performance vs release-distance (delta curves) --------------------

curves = []
for final_db in sorted(set(agg["final_database"].tolist())):
    sub = agg[agg["final_database"] == final_db].copy()
    if sub.empty:
        continue
    sub["delta"] = sub["to_release"].astype(int) - sub["from_release"].astype(int)
    # Use true-match fraction (TDM total) as the most honest external-matching metric.
    sub["metric"] = sub["frac_tdm_total"]
    curves.append(sub)

curves_df = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()
if curves_df.empty:
    print("No data for delta curves.")
else:
    # Aggregate over all (from,to) pairs at the same delta.
    delta_agg = (
        curves_df.groupby(["final_database", "delta"], as_index=False)["metric"]
        .agg([("mean", "mean"), ("std", "std"), ("n", "count")])
        .reset_index()
    )
    delta_agg["sem"] = delta_agg["std"] / delta_agg["n"].apply(lambda x: math.sqrt(x) if x else np.nan)

    # Export as a simple CSV (easy to turn into LaTeX later)
    out_csv = MANUSCRIPT_TABLES / "time_travel_matrix_delta_summary.csv"
    atomic_write_dataframe_csv(delta_agg, out_csv, index=False)
    atomic_write_dataframe_csv(delta_agg, EXPERIMENT_TABLES / out_csv.name, index=False)
    print("Wrote:", out_csv)

    fig2, ax = plt.subplots(1, 1, figsize=(8.5, 4.2), constrained_layout=True)
    for final_db, dsub in delta_agg.groupby("final_database"):
        ax.errorbar(
            dsub["delta"],
            dsub["mean"],
            yerr=dsub["sem"],
            fmt="-o",
            ms=3,
            lw=1.3,
            label=str(final_db),
        )
    ax.set_ylim(0, 1)
    ax.set_xlabel("Δ release = to_release − from_release")
    ax.set_ylabel("Fraction with any true match (TDM total)")
    ax.set_title("Time travel performance vs release distance")
    ax.legend(frameon=True, ncol=1)

    written2 = save_figure(fig2, "fig_time_travel_delta_curves.pdf", ctx, formats=("pdf",))
    print("Saved:", written2["pdf"])


# Deep-dive: what time travel reveals (marketing appendix)

The square matrix is not just “performance over time”. It lets you market a stronger claim without overclaiming accuracy:

1. **Direction matters (irreversibility):** mapping old→new is not equivalent to new→old.
2. **The time axis induces structure:** strong diagonals (small drift) and systematic off-diagonal degradation.
3. **External matching is a second axis:** mapping to HGNC/UniProt changes the ambiguity/fallback profile.

The remaining cells add cache-only diagnostics:

- Asymmetry heatmaps (forward − backward)
- Runtime heatmaps (seconds per conversion)
- Release difficulty curves (source vs target)
- “Best/worst pairs” table for quick manuscript insertion


In [ ]:
# -------------------- Derived metrics used by deep-dive panels --------------------

required_cols = {
    "final_database",
    "from_release",
    "to_release",
    "frac_tdm_total",
    "frac_1_to_0",
    "frac_1_to_n_total",
    "frac_atm_total",
    "seconds",
    "total",
}
missing = sorted(required_cols - set(agg.columns))
if missing:
    raise KeyError(f"Cached grid is missing required columns: {missing}")

agg2 = agg.copy()

# Success = not 1→0
agg2["frac_success"] = 1.0 - agg2["frac_1_to_0"]

# Conditional ambiguity: among successful queries, how often do we get 1→n?
agg2["frac_1_to_n_given_success"] = agg2["frac_1_to_n_total"] / agg2["frac_success"].replace(0, np.nan)

# Conditional fallback: among successful queries, how often do we fall back to ATM?
agg2["frac_atm_given_success"] = agg2["frac_atm_total"] / agg2["frac_success"].replace(0, np.nan)

# Runtime normalization
agg2["sec_per_id"] = agg2["seconds"] / agg2["total"].replace(0, np.nan)

primary_final_dbs = [
    db for db in ("Ensembl gene", "HGNC Symbol", "UniProtKB/Swiss-Prot") if db in set(agg2["final_database"])
]
if not primary_final_dbs:
    primary_final_dbs = sorted(set(agg2["final_database"]))

display(
    agg2[[
        "final_database",
        "from_release",
        "to_release",
        "frac_tdm_total",
        "frac_atm_total",
        "frac_1_to_n_total",
        "sec_per_id",
    ]].head()
)


In [ ]:
# -------------------- Figure: asymmetry (forward − backward) --------------------

def _square(mat: pd.DataFrame) -> pd.DataFrame:
    common = sorted(set(mat.index) & set(mat.columns))
    if not common:
        raise ValueError("No overlapping releases between index/columns; expected a square grid")
    return mat.loc[common, common]


def _pivot_metric(metric: str, *, final_db: str) -> pd.DataFrame:
    sub = agg2[agg2["final_database"] == final_db]
    if sub.empty:
        raise ValueError(f"No rows for final_database={final_db!r}")
    mat = sub.pivot(index="from_release", columns="to_release", values=metric)
    return mat.sort_index().sort_index(axis=1)


def _heatmap_centered(ax, mat: pd.DataFrame, *, title: str, vmax: float = 0.35):
    if sns is not None:
        sns.heatmap(
            mat,
            ax=ax,
            cmap="coolwarm",
            center=0,
            vmin=-vmax,
            vmax=vmax,
            cbar=False,
            square=True,
            linewidths=0.25,
            linecolor=MANUSCRIPT_COLORS["grid"],
        )
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    else:
        im = ax.imshow(mat.values, cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.figure.colorbar(im, ax=ax, fraction=0.046)
        ax.set_xticks(np.arange(len(mat.columns)))
        ax.set_yticks(np.arange(len(mat.index)))
        ax.set_xticklabels([str(c) for c in mat.columns], rotation=45, ha="right")
        ax.set_yticklabels([str(i) for i in mat.index], rotation=0)

    ax.set_title(title)
    ax.set_xlabel("to_release")
    ax.set_ylabel("from_release")


metrics_asym = [
    ("frac_tdm_total", "Any true match (TDM total)"),
    ("frac_1_to_0", "Failure (1→0)"),
    ("frac_1_to_n_given_success", "Ambiguity | success"),
    ("frac_atm_given_success", "Fallback | success"),
]

figA, axesA = plt.subplots(
    len(primary_final_dbs),
    len(metrics_asym),
    figsize=(4.4 * len(metrics_asym), 4.0 * len(primary_final_dbs)),
    constrained_layout=True,
)
if len(primary_final_dbs) == 1:
    axesA = np.array([axesA])

for r, final_db in enumerate(primary_final_dbs):
    for c, (metric, title) in enumerate(metrics_asym):
        mat = _square(_pivot_metric(metric, final_db=final_db))
        asym = mat - mat.T
        _heatmap_centered(axesA[r, c], asym, title=f"{final_db} — {title}\n(forward − backward)")

writtenA = save_figure(figA, "fig_time_travel_asymmetry_human.pdf", ctx, formats=("pdf",))
print("Saved:", writtenA["pdf"])


In [ ]:
# -------------------- Figure: runtime heatmaps (seconds per conversion) --------------------

def _heatmap_log(ax, mat: pd.DataFrame, *, title: str):
    x = mat.replace(0, np.nan).copy()
    x = np.log10(x)
    if sns is not None:
        sns.heatmap(
            x,
            ax=ax,
            cmap="mako",
            cbar=True,
            square=True,
            linewidths=0.25,
            linecolor=MANUSCRIPT_COLORS["grid"],
        )
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    else:
        im = ax.imshow(x.values, cmap="mako")
        ax.figure.colorbar(im, ax=ax, fraction=0.046)
        ax.set_xticks(np.arange(len(x.columns)))
        ax.set_yticks(np.arange(len(x.index)))
        ax.set_xticklabels([str(c) for c in x.columns], rotation=45, ha="right")
        ax.set_yticklabels([str(i) for i in x.index], rotation=0)

    ax.set_title(title + "\n(log10 seconds / conversion)")
    ax.set_xlabel("to_release")
    ax.set_ylabel("from_release")


figR, axesR = plt.subplots(
    1,
    len(primary_final_dbs),
    figsize=(6.3 * len(primary_final_dbs), 5.3),
    constrained_layout=True,
)
if len(primary_final_dbs) == 1:
    axesR = [axesR]

for ax, final_db in zip(axesR, primary_final_dbs):
    mat = _square(_pivot_metric("sec_per_id", final_db=final_db))
    _heatmap_log(ax, mat, title=f"{final_db}")

writtenR = save_figure(figR, "fig_time_travel_runtime_human.pdf", ctx, formats=("pdf",))
print("Saved:", writtenR["pdf"])


In [ ]:
# -------------------- Figure: release difficulty (source vs target) --------------------

def _difficulty(d: pd.DataFrame, *, group_col: str, metrics: list[str]) -> pd.DataFrame:
    g = d.groupby(["final_database", group_col], as_index=False)[metrics].mean(numeric_only=True)
    return g.sort_values(["final_database", group_col]).reset_index(drop=True)


metricsD = ["frac_tdm_total", "frac_1_to_0", "frac_1_to_n_total", "frac_atm_total"]
by_from = _difficulty(agg2, group_col="from_release", metrics=metricsD)
by_to = _difficulty(agg2, group_col="to_release", metrics=metricsD)

figD, axesD = plt.subplots(
    len(primary_final_dbs),
    2,
    figsize=(14, 4.2 * len(primary_final_dbs)),
    constrained_layout=True,
)
if len(primary_final_dbs) == 1:
    axesD = np.array([axesD])

for r, final_db in enumerate(primary_final_dbs):
    left = axesD[r, 0]
    right = axesD[r, 1]

    s_from = by_from[by_from["final_database"] == final_db]
    s_to = by_to[by_to["final_database"] == final_db]

    for ax, s, xcol, title in [
        (left, s_from, "from_release", "Source-release difficulty (mean over targets)"),
        (right, s_to, "to_release", "Target-release brittleness (mean over sources)"),
    ]:
        ax.plot(s[xcol], s["frac_tdm_total"], "-o", ms=3, lw=1.4, label="TDM total")
        ax.plot(s[xcol], s["frac_1_to_0"], "-o", ms=3, lw=1.4, label="1→0")
        ax.plot(s[xcol], s["frac_1_to_n_total"], "-o", ms=3, lw=1.4, label="1→n total")
        ax.plot(s[xcol], s["frac_atm_total"], "-o", ms=3, lw=1.4, label="ATM total")
        ax.set_ylim(0, 1)
        ax.set_title(f"{final_db} — {title}")
        ax.set_xlabel(xcol)
        ax.set_ylabel("Fraction")
        ax.legend(frameon=True, ncol=2, fontsize=9)

writtenD = save_figure(figD, "fig_time_travel_release_difficulty_human.pdf", ctx, formats=("pdf",))
print("Saved:", writtenD["pdf"])


In [ ]:
# -------------------- Export: best/worst release-pairs (table) --------------------

rows = []
for final_db, sub in agg2.groupby("final_database"):
    x = sub.copy()
    x["abs_delta"] = (x["to_release"].astype(int) - x["from_release"].astype(int)).abs()

    # Worst pairs: low TDM total (prefer larger deltas for interpretability)
    worst = x.sort_values(["frac_tdm_total", "abs_delta"], ascending=[True, False]).head(12)
    for _, r in worst.iterrows():
        rows.append(
            {
                "final_database": final_db,
                "kind": "worst_by_tdm_total",
                "from_release": int(r["from_release"]),
                "to_release": int(r["to_release"]),
                "abs_delta": int(r["abs_delta"]),
                "frac_tdm_total": float(r["frac_tdm_total"]),
                "frac_1_to_0": float(r["frac_1_to_0"]),
                "frac_1_to_n_total": float(r["frac_1_to_n_total"]),
                "frac_atm_total": float(r["frac_atm_total"]),
                "sec_per_id": float(r.get("sec_per_id", np.nan)),
            }
        )

    # Best pairs: high TDM total
    best = x.sort_values(["frac_tdm_total", "abs_delta"], ascending=[False, True]).head(12)
    for _, r in best.iterrows():
        rows.append(
            {
                "final_database": final_db,
                "kind": "best_by_tdm_total",
                "from_release": int(r["from_release"]),
                "to_release": int(r["to_release"]),
                "abs_delta": int(r["abs_delta"]),
                "frac_tdm_total": float(r["frac_tdm_total"]),
                "frac_1_to_0": float(r["frac_1_to_0"]),
                "frac_1_to_n_total": float(r["frac_1_to_n_total"]),
                "frac_atm_total": float(r["frac_atm_total"]),
                "sec_per_id": float(r.get("sec_per_id", np.nan)),
            }
        )

extremes = pd.DataFrame(rows)

out_ext = MANUSCRIPT_TABLES / "time_travel_pair_extremes.csv"
atomic_write_dataframe_csv(extremes, out_ext, index=False)
atomic_write_dataframe_csv(extremes, EXPERIMENT_TABLES / out_ext.name, index=False)

print("Wrote:", out_ext)
display(extremes.head(20))


# Marketing extension: directionality + stability indices

The square heatmaps already show that *time matters*. This section adds two reviewer-proof diagnostics that are easy to explain in a Results paragraph:

1. **Directionality (past vs future):** mapping is not perfectly symmetric; some changes are effectively irreversible.
2. **Release stability indices:** some releases are systematically “harder” as sources or targets (higher ambiguity / higher 1→0).

These plots are especially useful as **marketing** because they highlight a dimension that point-in-time mappers cannot even report: *a measurable time axis.*


In [ ]:
from time_travel_analysis import directional_distance_curve, release_stability_index  # noqa: E402
from experiments_utils import label_panels  # noqa: E402

metric = "frac_tdm_total"  # “success into the final target database” (TDM + explicit ambiguity)

# Use the same primary DBs as the main heatmap figure when available.
final_dbs = [db for db in ("Ensembl gene", "HGNC Symbol") if db in set(agg2["final_database"])]
if not final_dbs:
    final_dbs = sorted(set(agg2["final_database"]))[:2]

curve_rows = []
stab_rows = []
for final_db in final_dbs:
    c = directional_distance_curve(agg2, metric=metric, final_database=final_db)
    if not c.empty:
        c["final_database"] = final_db
        curve_rows.append(c)

    for axis in ("from", "to"):
        s = release_stability_index(agg2, metric=metric, final_database=final_db, axis=axis)
        if not s.empty:
            s["final_database"] = final_db
            stab_rows.append(s)

curves = pd.concat(curve_rows, ignore_index=True) if curve_rows else pd.DataFrame()
stability = pd.concat(stab_rows, ignore_index=True) if stab_rows else pd.DataFrame()

# Export stability indices as a manuscript-friendly CSV.
if not stability.empty:
    out_stab = MANUSCRIPT_TABLES / "time_travel_release_stability_index.csv"
    atomic_write_dataframe_csv(stability, out_stab, index=False)
    atomic_write_dataframe_csv(stability, EXPERIMENT_TABLES / out_stab.name, index=False)
    print("Wrote:", out_stab)

# Multi-panel: (a,b) directionality curves; (c,d) stability indices
fig, axes = plt.subplots(2, 2, figsize=(13.2, 7.6), constrained_layout=True)

direction_colors = {
    "future_or_same": MANUSCRIPT_COLORS["1→1"],
    "past": MANUSCRIPT_COLORS["1→0"],
}

for j, final_db in enumerate(final_dbs[:2]):
    ax = axes[0, j]
    sub = curves[curves["final_database"] == final_db]
    if sub.empty:
        ax.axis("off")
        ax.text(0.5, 0.5, f"No curve data for {final_db}", ha="center", va="center")
        continue
    for direction in ("future_or_same", "past"):
        d = sub[sub["direction"] == direction]
        if d.empty:
            continue
        ax.plot(
            d["abs_delta"],
            d[metric],
            "-o",
            lw=1.4,
            ms=3,
            label=direction.replace("_", " "),
            color=direction_colors.get(direction),
        )
    ax.set_ylim(0, 1)
    ax.set_xlabel("|to_release - from_release|")
    ax.set_ylabel("Mean TDM fraction")
    ax.set_title(f"{final_db} — directionality vs release distance")
    ax.legend(frameon=True, fontsize=9)

    ax2 = axes[1, j]
    s_from = stability[(stability["final_database"] == final_db) & (stability["axis"] == "from")]
    s_to = stability[(stability["final_database"] == final_db) & (stability["axis"] == "to")]
    if s_from.empty and s_to.empty:
        ax2.axis("off")
        ax2.text(0.5, 0.5, f"No stability data for {final_db}", ha="center", va="center")
        continue
    if not s_from.empty:
        ax2.plot(s_from["release"], s_from[metric], "-o", lw=1.2, ms=3, label="source", color=MANUSCRIPT_COLORS["IDTrack"])
    if not s_to.empty:
        ax2.plot(s_to["release"], s_to[metric], "-o", lw=1.2, ms=3, label="target", color=MANUSCRIPT_COLORS["1→n"])
    ax2.set_ylim(0, 1)
    ax2.set_xlabel("release")
    ax2.set_ylabel("Mean TDM fraction")
    ax2.set_title(f"{final_db} — release stability index")
    ax2.legend(frameon=True, fontsize=9)

label_panels(axes.ravel())

written = save_figure(fig, "fig_time_travel_directionality_stability_human.pdf", ctx, formats=("pdf",))
print("Saved:", written["pdf"])
